# NimbusPay Support Ticket Triage — LLM Fine-Tuning

**VIMA3YA | Machine Learning Intern — LLM Fine-Tuning Assignment**

This notebook preserves the completed audit and cleaning work from the uploaded notebook, fixes the baseline generation bug, and adds the remaining required workflow: baselines, FP16 LoRA, QLoRA, controlled experiments, dev evaluation, error analysis, test predictions, and submission files.

**Model:** `Qwen/Qwen2.5-1.5B-Instruct` (1.5B parameters)  
**Hardware target:** Google Colab T4  
**Training method:** LoRA / QLoRA only; assistant-only causal-LM loss  
**Important:** Run from top to bottom in a fresh Colab T4 runtime. The old FP16 training output is preserved as historical evidence, but a runtime reset removes the adapter files, so the training section must be run again to recreate them. No metric is invented in this notebook; all new results are generated when cells execute.

**Result integrity:** historical baseline and FP16 figures from the prior run are explicitly labelled in this notebook. No QLoRA, controlled-experiment, error-analysis, or hidden-test result is fabricated; those require execution in Colab.


In [ ]:
# 1. Upload and extract the candidate pack
from google.colab import files
import os, zipfile

uploaded = files.upload()
zip_files = [name for name in uploaded if name.lower().endswith(".zip")]
if not zip_files:
    raise FileNotFoundError("Upload the VIMA3YA training_assessment.zip file.")

ZIP_PATH = os.path.join("/content", zip_files[0])
EXTRACT_PATH = "/content/nimbuspay_assignment"
os.makedirs(EXTRACT_PATH, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_PATH)

# Detect the folder containing the assignment files (candidate_pack or archive root).
matches = []
for root, dirs, filenames in os.walk(EXTRACT_PATH):
    if {"score.py", "SCHEMA.md", "data"}.issubset(set(filenames) | set(dirs)):
        matches.append(root)

if not matches:
    raise FileNotFoundError("Could not locate a folder containing score.py, SCHEMA.md and data/.")

BASE = matches[0]
DATA = os.path.join(BASE, "data")
print("Assignment folder:", BASE)
print("Dataset files:", sorted(os.listdir(DATA)))

## Part A — Dataset audit and cleaning (preserved from your uploaded work)

In [ ]:
import json
import os
from collections import Counter

BASE = BASE if "BASE" in globals() else "/content/nimbuspay_assignment/candidate_pack"
DATA = os.path.join(BASE, "data")

files = {
    "train": os.path.join(DATA, "train.jsonl"),
    "dev": os.path.join(DATA, "dev.jsonl"),
    "test": os.path.join(DATA, "test_inputs.jsonl")
}

datasets = {}

for name, path in files.items():
    with open(path, "r", encoding="utf-8") as f:
        datasets[name] = [line.strip() for line in f if line.strip()]

    print(f"{name.upper()} records: {len(datasets[name])}")

print("\n--- TRAINING DATA JSON AUDIT ---")

valid_count = 0
invalid_count = 0
key_counts = Counter()
invalid_examples = []

for i, line in enumerate(datasets["train"]):
    try:
        record = json.loads(line)
        valid_count += 1

        # Inspect the assistant's labelled output
        messages = record.get("messages", [])
        assistant = next(
            (m for m in messages if m.get("role") == "assistant"),
            None
        )

        if assistant:
            try:
                output = json.loads(assistant["content"])
                key_counts[tuple(sorted(output.keys()))] += 1
            except Exception:
                invalid_count += 1
                invalid_examples.append(i + 1)

    except Exception:
        invalid_count += 1
        invalid_examples.append(i + 1)

print("Valid training records:", valid_count)
print("Invalid records or assistant labels:", invalid_count)

print("\nLabel key combinations:")
for keys, count in key_counts.items():
    print(count, "records:", keys)

print("\nFirst 5 training examples:")
for line in datasets["train"][:5]:
    print(json.dumps(json.loads(line), indent=2, ensure_ascii=False))
    print("-" * 50)

TRAIN records: 1445
DEV records: 200
TEST records: 400

--- TRAINING DATA JSON AUDIT ---
Valid training records: 1445
Invalid records or assistant labels: 25

Label key combinations:
1350 records: ('amount_paise', 'category', 'channel', 'language', 'needs_human', 'priority', 'txn_date', 'txn_id')
70 records: ('amount_paise', 'category', 'channel', 'language', 'needs_human', 'prio', 'txn_date', 'txn_id')

First 5 training examples:
{
  "id": "tr-00001",
  "messages": [
    {
      "role": "system",
      "content": "You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON."
    },
    {
      "role": "user",
      "content": "Received: 2026-07-25 (Sat)\nSource: app-chat\n\nHi, I have a top up that didn't go through. Why? It was today. Mode: internet banking. My phone is a Redmi Note 12. It is 4.7k in total. My account balance is 258 rupees right now. The txn id is NP1599046332. Waiting for your reply."
    },
    {
      "role": "assistant",
      "content": "

In [ ]:
import json

train_path = files["train"]

invalid_labels = []
wrong_key_records = []

with open(train_path, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        messages = record.get("messages", [])

        user_msg = next(
            (m["content"] for m in messages if m["role"] == "user"),
            ""
        )
        assistant_msg = next(
            (m["content"] for m in messages if m["role"] == "assistant"),
            ""
        )

        try:
            label = json.loads(assistant_msg)

            if "prio" in label and "priority" not in label:
                wrong_key_records.append({
                    "id": record["id"],
                    "user": user_msg,
                    "label": label
                })

        except json.JSONDecodeError:
            invalid_labels.append({
                "id": record["id"],
                "user": user_msg,
                "raw_label": assistant_msg
            })

print("INVALID LABELS:", len(invalid_labels))
print("WRONG KEY RECORDS:", len(wrong_key_records))

print("\n========== INVALID LABELS ==========")
for item in invalid_labels:
    print("\nID:", item["id"])
    print("TICKET:", item["user"])
    print("RAW LABEL:", item["raw_label"])

print("\n========== WRONG KEY EXAMPLES ==========")
for item in wrong_key_records[:5]:
    print("\nID:", item["id"])
    print("LABEL:", item["label"])

INVALID LABELS: 25
WRONG KEY RECORDS: 70

========== INVALID LABELS ==========

ID: tr-00060
TICKET: Received: 2026-09-17 (Thu)
Source: email

Subject: Complaint

Hi team,
Everytime I try to prove my identity it won't let me. Bank ref no 974846156473.

Regards,
Ananya Bose
RAW LABEL: {"category": "kyc", "priority": "P3", "amount_paise": null, "txn_date": null, "t

ID: tr-00095
TICKET: Received: 2026-09-15 (Tue)
Source: app-chat

Dear NimbusPay support, May I have another card? My phone is a Realme 11. Thanks.
RAW LABEL: {"category": "other", "priority": "P3", "amount_paise": null, "txn_date": null, "txn_id": null, "c

ID: tr-00125
TICKET: Received: 2026-08-18 (Tue)
Source: app-chat

Hey, My card payment didn't work. Dated 12-08-26. It is Rs. 3925 in total. The txn id is NP6948115654. My earlier ticket number is #99485. Kindly resolve this asap.
RAW LABEL: {"category": "payment_failed", "priority": "P3", "amount_paise": 392500, "txn_date": "2026-08-12", "t

ID: tr-00172
TICKET: Received

In [ ]:
import json

invalid_details = []

with open(files["train"], "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)

        messages = record["messages"]
        assistant = next(
            m["content"] for m in messages
            if m["role"] == "assistant"
        )

        try:
            json.loads(assistant)
        except json.JSONDecodeError as e:
            user = next(
                m["content"] for m in messages
                if m["role"] == "user"
            )

            invalid_details.append({
                "id": record["id"],
                "ticket": user,
                "raw_label": assistant,
                "length": len(assistant),
                "error": str(e)
            })

print("Total malformed labels:", len(invalid_details))

for item in invalid_details:
    print("\n" + "=" * 70)
    print("ID:", item["id"])
    print("LABEL LENGTH:", item["length"])
    print("JSON ERROR:", item["error"])
    print("TICKET:\n", item["ticket"])
    print("COMPLETE RAW LABEL:\n", repr(item["raw_label"]))

Total malformed labels: 25

ID: tr-00060
LABEL LENGTH: 80
JSON ERROR: Unterminated string starting at: line 1 column 79 (char 78)
TICKET:
 Received: 2026-09-17 (Thu)
Source: email

Subject: Complaint

Hi team,
Everytime I try to prove my identity it won't let me. Bank ref no 974846156473.

Regards,
Ananya Bose
COMPLETE RAW LABEL:
 '{"category": "kyc", "priority": "P3", "amount_paise": null, "txn_date": null, "t'

ID: tr-00095
LABEL LENGTH: 98
JSON ERROR: Unterminated string starting at: line 1 column 97 (char 96)
TICKET:
 Received: 2026-09-15 (Tue)
Source: app-chat

Dear NimbusPay support, May I have another card? My phone is a Realme 11. Thanks.
COMPLETE RAW LABEL:
 '{"category": "other", "priority": "P3", "amount_paise": null, "txn_date": null, "txn_id": null, "c'

ID: tr-00125
LABEL LENGTH: 101
JSON ERROR: Unterminated string starting at: line 1 column 100 (char 99)
TICKET:
 Received: 2026-08-18 (Tue)
Source: app-chat

Hey, My card payment didn't work. Dated 12-08-26. It is Rs. 3925

In [ ]:
import json
import os
import csv

# Required output fields, in the required order
FIELDS = [
    "category", "priority", "amount_paise", "txn_date",
    "txn_id", "channel", "language", "needs_human"
]

# Corrections reconstructed from each ticket
repairs = {
    "tr-00060": ["kyc", "P3", None, None, None, None, "en", False],
    "tr-00095": ["other", "P3", None, None, None, None, "en", False],
    "tr-00125": ["payment_failed", "P3", 392500, "2026-08-12", "NP6948115654", "card", "en", False],
    "tr-00172": ["kyc", "P3", None, None, None, None, "en", False],
    "tr-00267": ["kyc", "P3", None, None, None, None, "en", False],
    "tr-00357": ["account_access", "P2", None, None, None, None, "en", False],
    "tr-00478": ["refund", "P1", 16041600, None, "NP1893758181", None, "en", True],
    "tr-00497": ["payment_failed", "P3", 242300, "2026-06-25", None, None, "en", False],
    "tr-00558": ["payment_failed", "P1", 15137000, "2026-07-30", "NP8621659965", "netbanking", "en", True],
    "tr-00605": ["payment_failed", "P3", 37700, "2026-07-25", None, None, "en", False],
    "tr-00674": ["payment_failed", "P3", 338500, "2026-09-16", None, "upi", "en", False],
    "tr-00735": ["payment_failed", "P1", 29687900, None, "NP7764977551", "netbanking", "en", True],
    "tr-00765": ["refund", "P3", 13600, "2026-06-27", "NP4193555354", "card", "hi-en", False],
    "tr-00809": ["payment_failed", "P3", 97000, "2026-07-05", "NP8738912683", "card", "hi-en", False],
    "tr-00824": ["offers", "P3", 27100, None, None, None, "hi-en", False],
    "tr-00846": ["payment_failed", "P3", 23000, None, "NP4657109851", "wallet", "en", False],
    "tr-00958": ["refund", "P3", 59700, None, None, "wallet", "en", True],
    "tr-00960": ["account_access", "P1", None, None, None, None, "hi-en", True],
    "tr-00975": ["fraud", "P1", 94400, "2026-08-29", None, "card", "en", True],
    "tr-00978": ["account_access", "P2", None, None, None, None, "en", False],
    "tr-00979": ["payment_failed", "P3", 83325, "2026-08-07", "NP7335161087", "card", "en", True],
    "tr-00980": ["other", "P3", None, None, None, None, "en", False],
    "tr-01070": ["payment_failed", "P2", 1787400, "2026-08-11", "NP2339235751", "card", "en", False],
    "tr-01087": ["offers", "P3", None, None, None, "card", "en", False],
    "tr-01199": ["payment_failed", "P3", 141099, None, None, "upi", "en", False]
}

cleaned_records = []
cleaning_log = []

with open(files["train"], "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        record_id = record["id"]

        if record_id in repairs:
            new_label = dict(zip(FIELDS, repairs[record_id]))

            assistant = next(
                m for m in record["messages"]
                if m["role"] == "assistant"
            )

            assistant["content"] = json.dumps(
                new_label, ensure_ascii=False
            )

            cleaning_log.append({
                "id": record_id,
                "action": "reconstructed_truncated_label",
                "reason": "Original assistant JSON was truncated; label reconstructed from ticket and schema."
            })

        cleaned_records.append(record)

# Save a separate cleaned dataset
cleaned_path = os.path.join(BASE, "train_reconstructed.jsonl")

with open(cleaned_path, "w", encoding="utf-8") as f:
    for record in cleaned_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Reconstructed labels:", len(cleaning_log))
print("Cleaned dataset:", cleaned_path)
print("Original dataset preserved.")

Reconstructed labels: 25
Cleaned dataset: /content/nimbuspay_assignment/candidate_pack/train_reconstructed.jsonl
Original dataset preserved.


In [ ]:
import json

errors = []
seen = set()

with open(cleaned_path, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)

        assistant = next(
            m["content"] for m in record["messages"]
            if m["role"] == "assistant"
        )

        try:
            label = json.loads(assistant)

            if list(label.keys()) != FIELDS:
                errors.append((record["id"], "Incorrect fields"))

            if record["id"] in repairs:
                seen.add(record["id"])

        except Exception as e:
            errors.append((record["id"], str(e)))

print("Reconstructed IDs found:", len(seen))
print("Validation errors:", len(errors))
print("Validation details:", errors[:10])

Reconstructed IDs found: 25
Validation errors: 70
Validation details: [('tr-00020', 'Incorrect fields'), ('tr-00049', 'Incorrect fields'), ('tr-00058', 'Incorrect fields'), ('tr-00072', 'Incorrect fields'), ('tr-00077', 'Incorrect fields'), ('tr-00114', 'Incorrect fields'), ('tr-00120', 'Incorrect fields'), ('tr-00123', 'Incorrect fields'), ('tr-00153', 'Incorrect fields'), ('tr-00160', 'Incorrect fields')]


In [ ]:
import json

wrong_priority_records = []

with open(cleaned_path, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)

        assistant = next(
            m["content"] for m in record["messages"]
            if m["role"] == "assistant"
        )

        label = json.loads(assistant)

        if "prio" in label:
            ticket = next(
                m["content"] for m in record["messages"]
                if m["role"] == "user"
            )

            wrong_priority_records.append({
                "id": record["id"],
                "ticket": ticket,
                "label": label
            })

print("Records needing priority correction:",
      len(wrong_priority_records))

for item in wrong_priority_records:
    print("\n" + "=" * 65)
    print("ID:", item["id"])
    print("TICKET:", item["ticket"])
    print("LABEL:", item["label"])

Records needing priority correction: 70

ID: tr-00020
TICKET: Received: 2026-09-05 (Sat)
Source: ivr-transcript

[IVR transcript] Hey, How can I check if I have received a refund if it is not yet on my statement? 978 rupees is the amount. I have contacted support about this before. Mode: wallet. Using the Android app. This happened on 02-09-26. Thanks.
LABEL: {'category': 'refund', 'prio': 'low', 'amount_paise': 97800, 'txn_date': '02/09/2026', 'txn_id': None, 'channel': 'wallet', 'language': 'en', 'needs_human': True}

ID: tr-00049
TICKET: Received: 2026-09-01 (Tue)
Source: ivr-transcript

[IVR transcript]  Regsitered email id kaise change karun? Dhanyavaad.
LABEL: {'category': 'other', 'prio': 'low', 'amount_paise': None, 'txn_date': None, 'txn_id': None, 'channel': None, 'language': 'hi-en', 'needs_human': False}

ID: tr-00058
TICKET: Received: 2026-09-13 (Sun)
Source: ivr-transcript

[IVR transcript] Dear NimbusPay support, I tried to make a transfer and it didn't work. This happen

In [ ]:
import json
import os
import re
import tempfile
from datetime import datetime

# Use the cleaned training file from the previous steps
assert os.path.exists(cleaned_path), f"File not found: {cleaned_path}"

priority_map = {
    "low": "P3",
    "medium": "P2",
    "high": "P1",
    "P1": "P1",
    "P2": "P2",
    "P3": "P3"
}

legal_terms = [
    "rbi", "ombudsman", "consumer court",
    "lawyer", "legal notice", "legal action"
]

repeat_terms = [
    "third time", "fourth time", "already contacted",
    "contacted support before", "second complaint",
    "previous complaint", "written 4 times",
    "written 4 time", "baar baar", "teesri baar",
    "pehle bhi complaint", "pichla ticket"
]

corrected_count = 0
corrections = []

with open(cleaned_path, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

for record in records:
    assistant_msg = next(
        m for m in record["messages"] if m["role"] == "assistant"
    )
    user_msg = next(
        m for m in record["messages"] if m["role"] == "user"
    )

    label = json.loads(assistant_msg["content"])

    if "prio" not in label:
        continue

    old_label = label.copy()
    ticket = user_msg["content"].lower()

    # Rename old priority field and map its value
    old_priority = label.pop("prio")
    label["priority"] = priority_map.get(old_priority, "P3")

    # Convert DD/MM/YYYY to YYYY-MM-DD
    date_value = label.get("txn_date")
    if date_value:
        try:
            label["txn_date"] = datetime.strptime(
                date_value, "%d/%m/%Y"
            ).strftime("%Y-%m-%d")
        except ValueError:
            pass

    category = label["category"]
    amount = label.get("amount_paise")

    # Recalculate priority using assignment rules
    legal_threat = any(term in ticket for term in legal_terms)

    if (
        category == "fraud"
        or (amount is not None and amount >= 5_000_000)
        or legal_threat
    ):
        label["priority"] = "P1"

    elif (
        category == "account_access"
        or (
            category in ["payment_failed", "refund"]
            and amount is not None
            and amount >= 500_000
        )
    ):
        label["priority"] = "P2"

    else:
        label["priority"] = "P3"

    # Preserve existing escalation and detect repeated contact
    repeated_contact = any(term in ticket for term in repeat_terms)

    label["needs_human"] = (
        label.get("needs_human", False)
        or label["priority"] == "P1"
        or repeated_contact
    )

    assistant_msg["content"] = json.dumps(
        label, ensure_ascii=False
    )

    corrected_count += 1
    corrections.append(record["id"])

# Save updated training file
temp_path = cleaned_path + ".tmp"

with open(temp_path, "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

os.replace(temp_path, cleaned_path)

print("Corrected records:", corrected_count)
print("Expected:", 70)
print("Saved to:", cleaned_path)
print("Sample corrected IDs:", corrections[:10])

Corrected records: 70
Expected: 70
Saved to: /content/nimbuspay_assignment/candidate_pack/train_reconstructed.jsonl
Sample corrected IDs: ['tr-00020', 'tr-00049', 'tr-00058', 'tr-00072', 'tr-00077', 'tr-00114', 'tr-00120', 'tr-00123', 'tr-00153', 'tr-00160']


In [ ]:
import json

errors = []
count = 0

with open(cleaned_path, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        assistant = next(
            m["content"] for m in record["messages"]
            if m["role"] == "assistant"
        )
        label = json.loads(assistant)

        expected_keys = {
            "category", "priority", "amount_paise",
            "txn_date", "txn_id", "channel",
            "language", "needs_human"
        }

        if set(label.keys()) != expected_keys:
            errors.append((record["id"], "Incorrect fields"))

        if label.get("priority") not in ["P1", "P2", "P3"]:
            errors.append((record["id"], "Invalid priority"))

        count += 1

print("Total records:", count)
print("Validation errors:", len(errors))
print("First errors:", errors[:10])

Total records: 1445
Validation errors: 0
First errors: []


In [ ]:
import json
from collections import Counter

valid_categories = {
    "payment_failed", "refund", "fraud",
    "account_access", "kyc", "offers", "other"
}

valid_priorities = {"P1", "P2", "P3"}
valid_channels = {"upi", "card", "netbanking", "wallet", None}

errors = []
category_counts = Counter()

with open(cleaned_path, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        label = json.loads(next(
            m["content"] for m in record["messages"]
            if m["role"] == "assistant"
        ))

        rid = record["id"]
        category_counts[label["category"]] += 1

        if label["category"] not in valid_categories:
            errors.append((rid, "Invalid category"))

        if label["priority"] not in valid_priorities:
            errors.append((rid, "Invalid priority"))

        if label["channel"] not in valid_channels:
            errors.append((rid, "Invalid channel"))

        if label["amount_paise"] is not None and (
            not isinstance(label["amount_paise"], int)
            or isinstance(label["amount_paise"], bool)
            or label["amount_paise"] < 0
        ):
            errors.append((rid, "Invalid amount"))

        date = label["txn_date"]
        if date is not None:
            try:
                from datetime import datetime
                datetime.strptime(date, "%Y-%m-%d")
            except ValueError:
                errors.append((rid, "Invalid date"))

        txn_id = label["txn_id"]
        if txn_id is not None and not __import__("re").fullmatch(
            r"NP\d{10}", txn_id
        ):
            errors.append((rid, "Invalid transaction ID"))

        if label["category"] in {"account_access", "kyc", "other"}:
            fields = ["amount_paise", "txn_date", "txn_id", "channel"]
            if any(label[field] is not None for field in fields):
                errors.append((rid, "Unexpected transaction fields"))

print("Category distribution:", dict(category_counts))
print("Data-quality errors:", len(errors))
print("First 30 errors:", errors[:30])

Category distribution: {'payment_failed': 425, 'account_access': 186, 'refund': 315, 'fraud': 92, 'kyc': 168, 'offers': 155, 'other': 104}
Data-quality errors: 20
First 30 errors: [('tr-00071', 'Invalid transaction ID'), ('tr-00147', 'Invalid transaction ID'), ('tr-00151', 'Invalid transaction ID'), ('tr-00391', 'Invalid transaction ID'), ('tr-00420', 'Invalid transaction ID'), ('tr-00460', 'Invalid transaction ID'), ('tr-00470', 'Invalid transaction ID'), ('tr-00570', 'Invalid transaction ID'), ('tr-00595', 'Invalid transaction ID'), ('tr-00649', 'Invalid transaction ID'), ('tr-00771', 'Invalid transaction ID'), ('tr-00838', 'Invalid transaction ID'), ('tr-00888', 'Invalid transaction ID'), ('tr-00899', 'Invalid transaction ID'), ('tr-00908', 'Invalid transaction ID'), ('tr-01012', 'Invalid transaction ID'), ('tr-01136', 'Invalid transaction ID'), ('tr-01188', 'Invalid transaction ID'), ('tr-01238', 'Invalid transaction ID'), ('tr-01363', 'Invalid transaction ID')]


In [ ]:
import json

problem_ids = {
    "tr-00071", "tr-00147", "tr-00151", "tr-00391",
    "tr-00420", "tr-00460", "tr-00470", "tr-00570",
    "tr-00595", "tr-00649", "tr-00771", "tr-00838",
    "tr-00888", "tr-00899", "tr-00908", "tr-01012",
    "tr-01136", "tr-01188", "tr-01238", "tr-01363"
}

with open(cleaned_path, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)

        if record["id"] in problem_ids:
            ticket = next(
                m["content"] for m in record["messages"]
                if m["role"] == "user"
            )

            label = json.loads(next(
                m["content"] for m in record["messages"]
                if m["role"] == "assistant"
            ))

            print("\n" + "=" * 70)
            print("ID:", record["id"])
            print("TICKET:", ticket)
            print("CURRENT LABEL:", label)


ID: tr-00071
TICKET: Received: 2026-08-17 (Mon)
Source: email

Subject: Problem with NimbusPay

Hello ji,
Ek hi order ke do baar paise kat gaye, ek wapas kar do. Reference NP-9718716109 hai. Android app use kar raha hoon. 4 din pehle ki baat hai. Rs. 954 kat gaye. Reply ka wait kar raha hoon.

Regards,
Arjun Das
CURRENT LABEL: {'category': 'refund', 'priority': 'P3', 'amount_paise': 95400, 'txn_date': '2026-08-13', 'txn_id': 'NP-9718716109', 'channel': None, 'language': 'hi-en', 'needs_human': False}

ID: tr-00147
TICKET: Received: 2026-07-04 (Sat)
Source: app-chat

Sir, Ek hi order ke do baar paise kat gaye, ek wapas kar do. Jun 22 ki transaction hai. UPI se hua tha. np9891267518 check kar lo. Rs. 859 kat gaye. Mera phone iPhone 13 hai. Jaldi solve karo please.
CURRENT LABEL: {'category': 'refund', 'priority': 'P3', 'amount_paise': 85900, 'txn_date': '2026-06-22', 'txn_id': 'np9891267518', 'channel': 'upi', 'language': 'hi-en', 'needs_human': False}

ID: tr-00151
TICKET: Received: 20

In [ ]:
import json
import re
import os

corrections = {
    "tr-00071": "NP9718716109",
    "tr-00147": "NP9891267518",
    "tr-00151": "NP2339259875",
    "tr-00391": "NP9255627510",
    "tr-00420": "NP8163347892",
    "tr-00460": "NP6949410784",
    "tr-00470": "NP1124546709",
    "tr-00570": "NP8726797611",
    "tr-00595": "NP3247474643",
    "tr-00649": "NP8919677483",
    "tr-00771": "NP9038808761",
    "tr-00838": "NP6186175115",
    "tr-00888": "NP7786684387",
    "tr-00899": "NP1325500121",
    "tr-00908": "NP4585696868",
    "tr-01012": "NP7854905014",
    "tr-01136": "NP5804974985",
    "tr-01188": "NP0530288495",
    "tr-01238": "NP7739480420",
    "tr-01363": "NP2558643472"
}

with open(cleaned_path, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

updated = 0

for record in records:
    if record["id"] in corrections:
        assistant = next(
            m for m in record["messages"]
            if m["role"] == "assistant"
        )
        label = json.loads(assistant["content"])
        label["txn_id"] = corrections[record["id"]]
        assistant["content"] = json.dumps(
            label, ensure_ascii=False
        )
        updated += 1

assert updated == 20, f"Expected 20 updates, got {updated}"

temp_path = cleaned_path + ".tmp"
with open(temp_path, "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

os.replace(temp_path, cleaned_path)

print("Transaction IDs corrected:", updated)
print("Training records preserved:", len(records))

Transaction IDs corrected: 20
Training records preserved: 1445


## Environment setup and sequence-length evidence

In [ ]:
import os

BASE = "/content/nimbuspay_assignment/candidate_pack"
DATA = os.path.join(BASE, "data")

print("Base folder exists:", os.path.exists(BASE))
print("Data folder exists:", os.path.exists(DATA))

if os.path.exists(DATA):
    print("\nDataset files:")
    for file in os.listdir(DATA):
        print(file)

Base folder exists: True
Data folder exists: True

Dataset files:
train.jsonl
test_inputs.jsonl
dev.jsonl


In [ ]:
import os
import json

BASE = "/content/nimbuspay_assignment/candidate_pack"

print("JSONL files found:\n")

for root, dirs, files in os.walk("/content/nimbuspay_assignment"):
    for file in files:
        if file.endswith(".jsonl") or file.endswith(".csv"):
            path = os.path.join(root, file)
            print(path, "|", os.path.getsize(path), "bytes")

print("\nOriginal dataset record counts:")

for filename in ["train.jsonl", "dev.jsonl", "test_inputs.jsonl"]:
    path = os.path.join(BASE, "data", filename)
    with open(path, "r", encoding="utf-8") as f:
        count = sum(1 for line in f if line.strip())
    print(filename, ":", count)

JSONL files found:

/content/nimbuspay_assignment/candidate_pack/train_reconstructed.jsonl | 1165242 bytes
/content/nimbuspay_assignment/candidate_pack/data/train.jsonl | 1163136 bytes
/content/nimbuspay_assignment/candidate_pack/data/test_inputs.jsonl | 236519 bytes
/content/nimbuspay_assignment/candidate_pack/data/dev.jsonl | 147085 bytes

Original dataset record counts:
train.jsonl : 1445
dev.jsonl : 200
test_inputs.jsonl : 400


In [ ]:
# Install a compatible, pinned FP16/LoRA stack.
# Keep the Colab-provided PyTorch build; do not reinstall or upgrade torch.
%pip uninstall -y torchao
%pip install -q transformers==4.46.3 peft==0.13.2 accelerate==1.2.1 safetensors sentencepiece pandas scikit-learn

import torch, transformers, peft, accelerate
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU before continuing.")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))

In [ ]:
# Load the Qwen tokenizer
from transformers import AutoTokenizer
import os

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

print("Tokenizer loaded:", MODEL_ID)
print("Vocabulary size:", len(tokenizer))
print("Chat template available:", tokenizer.chat_template is not None)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded: Qwen/Qwen2.5-1.5B-Instruct
Vocabulary size: 151665
Chat template available: True


In [ ]:
import json
import numpy as np

TRAIN_PATH = os.path.join(BASE, "train_reconstructed.jsonl")

lengths = []

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)

        messages = record["messages"]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False
        )

        token_ids = tokenizer(
            text,
            add_special_tokens=False
        )["input_ids"]

        lengths.append(len(token_ids))

print("Total records:", len(lengths))
print("Minimum:", min(lengths))
print("Median:", int(np.percentile(lengths, 50)))
print("90th percentile:", int(np.percentile(lengths, 90)))
print("95th percentile:", int(np.percentile(lengths, 95)))
print("99th percentile:", int(np.percentile(lengths, 99)))
print("Maximum:", max(lengths))

Total records: 1445
Minimum: 107
Median: 168
90th percentile: 214
95th percentile: 225
99th percentile: 1804
Maximum: 2215


In [ ]:
#Inspect long training examples
import json

long_records = []

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)

        text = tokenizer.apply_chat_template(
            record["messages"],
            tokenize=False
        )

        token_count = len(
            tokenizer(text, add_special_tokens=False)["input_ids"]
        )

        if token_count > 512:
            long_records.append({
                "id": record.get("id"),
                "tokens": token_count,
                "messages": record["messages"]
            })

print("Records above 512 tokens:", len(long_records))

for item in sorted(long_records, key=lambda x: x["tokens"], reverse=True)[:5]:
    print("\nID:", item["id"])
    print("Tokens:", item["tokens"])
    print("Messages:", item["messages"])

Records above 512 tokens: 40

ID: tr-00817
Tokens: 2215
Messages: [{'role': 'system', 'content': "You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON."}, {'role': 'user', 'content': 'Received: 2026-07-14 (Tue)\nSource: email\n\nSubject: Help needed\n\nHi,\nEk hi order ke do baar paise kat gaye, ek wapas kar do. Android app use kar raha hoon. Card se hua tha. Rakam ₹99,936 hai. Yeh 10-07-26 ko hua. Account mein abhi Rs 3,762.25 pade hain. Teesri baar bol raha hoon. Dhanyavaad.\n\nRegards,\nArjun Das\n\n> On 27 Jun 2026, NimbusPay Support wrote:\n> Thank you for contacting NimbusPay. Your request has been registered with ticket #26806. Our team will get back to you within 48 working hours. Please do not share your OTP, PIN or card details with anyone, including people who claim to be from NimbusPay. You can track the status of your request from the Help section of the app. If you have already received a resolution, please ignore this message. We value your

In [ ]:
#Check the assistant response lengths
assistant_lengths = []

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)

        for message in record["messages"]:
            if message["role"] == "assistant":
                count = len(tokenizer(
                    message["content"],
                    add_special_tokens=False
                )["input_ids"])
                assistant_lengths.append(count)

print("Assistant response 95th percentile:",
      int(np.percentile(assistant_lengths, 95)))
print("Maximum assistant response:",
      max(assistant_lengths))

Assistant response 95th percentile: 79
Maximum assistant response: 83


In [ ]:
#Create the final cleaned training file

import json
import os

SOURCE_PATH = os.path.join(BASE, "train_reconstructed.jsonl")
CLEAN_PATH = os.path.join(BASE, "train_cleaned.jsonl")

changed_records = []
total_removed_lines = 0

with open(SOURCE_PATH, "r", encoding="utf-8") as fin, \
     open(CLEAN_PATH, "w", encoding="utf-8") as fout:

    for line in fin:
        record = json.loads(line)

        for message in record["messages"]:
            if message["role"] == "user":
                original = message["content"]

                kept_lines = [
                    line for line in original.splitlines()
                    if not line.lstrip().startswith(">")
                ]

                cleaned = "\n".join(kept_lines).rstrip()

                if cleaned != original:
                    removed = len(original.splitlines()) - len(kept_lines)
                    total_removed_lines += removed
                    changed_records.append(record.get("id"))

                message["content"] = cleaned

        fout.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Final cleaned file:", CLEAN_PATH)
print("Records preserved:", 1445)
print("Records with quoted history removed:", len(changed_records))
print("Quoted lines removed:", total_removed_lines)
print("File exists:", os.path.exists(CLEAN_PATH))

Final cleaned file: /content/nimbuspay_assignment/candidate_pack/train_cleaned.jsonl
Records preserved: 1445
Records with quoted history removed: 63
Quoted lines removed: 1024
File exists: True


In [ ]:
#Recheck token lengths

import numpy as np

clean_lengths = []

with open(CLEAN_PATH, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        text = tokenizer.apply_chat_template(
            record["messages"],
            tokenize=False
        )
        ids = tokenizer(text, add_special_tokens=False)["input_ids"]
        clean_lengths.append(len(ids))

print("Total records:", len(clean_lengths))
print("Median:", int(np.percentile(clean_lengths, 50)))
print("95th percentile:", int(np.percentile(clean_lengths, 95)))
print("99th percentile:", int(np.percentile(clean_lengths, 99)))
print("Maximum:", max(clean_lengths))
print("Records above 512:", sum(x > 512 for x in clean_lengths))

Total records: 1445
Median: 167
95th percentile: 218
99th percentile: 235
Maximum: 273
Records above 512: 0


In [ ]:
#Verify the cleaned dataset
import json

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

original = load_jsonl(os.path.join(BASE, "train_reconstructed.jsonl"))
cleaned = load_jsonl(CLEAN_PATH)

assert len(original) == len(cleaned), "Record count mismatch!"

assistant_changes = 0
remaining_quotes = 0
id_changes = 0

for old, new in zip(original, cleaned):
    assert old["id"] == new["id"], "Record ID mismatch!"

    old_assistant = next(m["content"] for m in old["messages"] if m["role"] == "assistant")
    new_assistant = next(m["content"] for m in new["messages"] if m["role"] == "assistant")

    if old_assistant != new_assistant:
        assistant_changes += 1

    old_user = next(m["content"] for m in old["messages"] if m["role"] == "user")
    new_user = next(m["content"] for m in new["messages"] if m["role"] == "user")

    if any(line.lstrip().startswith(">") for line in new_user.splitlines()):
        remaining_quotes += 1

print("Original records:", len(original))
print("Cleaned records:", len(cleaned))
print("Assistant label changes during quote cleaning:", assistant_changes)
print("Records with remaining quoted lines:", remaining_quotes)
print("Dataset verification:", "PASSED" if assistant_changes == 0 and remaining_quotes == 0 else "CHECK REQUIRED")

Original records: 1445
Cleaned records: 1445
Assistant label changes during quote cleaning: 0
Records with remaining quoted lines: 0
Dataset verification: PASSED


In [ ]:
#Create the cleaning log

import json
import csv
import os
import re
from collections import Counter

RAW_PATH = os.path.join(DATA, "train.jsonl")
FINAL_PATH = CLEAN_PATH
LOG_PATH = os.path.join(BASE, "cleaning_log.csv")

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def get_message(record, role):
    return next(
        m["content"] for m in record["messages"]
        if m["role"] == role
    )

raw_records = read_jsonl(RAW_PATH)
final_records = read_jsonl(FINAL_PATH)

final_by_id = {r["id"]: r for r in final_records}
log_rows = []

for raw in raw_records:
    record_id = raw["id"]
    final = final_by_id[record_id]

    raw_assistant = get_message(raw, "assistant")
    final_assistant = get_message(final, "assistant")

    # 1. Detect malformed original JSON labels
    try:
        raw_label = json.loads(raw_assistant)
        raw_label_valid = True
    except (json.JSONDecodeError, TypeError):
        raw_label = None
        raw_label_valid = False

    if not raw_label_valid:
        log_rows.append([
            record_id, "fix",
            "Reconstructed malformed or truncated assistant JSON label using the assignment schema."
        ])

    final_label = json.loads(final_assistant)

    # 2. Detect legacy prio field
    if raw_label_valid and "prio" in raw_label and "priority" not in raw_label:
        log_rows.append([
            record_id, "fix",
            "Renamed legacy prio field to priority and normalized its value."
        ])

    # 3. Detect transaction ID normalization
    if raw_label_valid:
        old_txn = raw_label.get("txn_id")
        new_txn = final_label.get("txn_id")

        if old_txn != new_txn and isinstance(old_txn, str):
            normalized = re.sub(
                r"[^A-Za-z0-9]", "", old_txn
            ).upper()

            if normalized == new_txn and re.fullmatch(
                r"NP\d{10}", normalized
            ):
                log_rows.append([
                    record_id, "fix",
                    "Normalized transaction ID to uppercase NP followed by 10 digits."
                ])

    # 4. Detect quoted support-history removal
    raw_user = get_message(raw, "user")
    final_user = get_message(final, "user")

    removed_lines = [
        line for line in raw_user.splitlines()
        if line.lstrip().startswith(">")
    ]

    if removed_lines and raw_user != final_user:
        log_rows.append([
            record_id, "fix",
            f"Removed {len(removed_lines)} quoted support-history lines while preserving the customer's message."
        ])

# Save CSV
with open(LOG_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "action", "reason"])
    writer.writerows(log_rows)

print("Cleaning log created:", LOG_PATH)
print("Total log entries:", len(log_rows))
print("Unique records documented:", len(set(row[0] for row in log_rows)))
print("\nReason counts:")
print(Counter(row[2].split()[0] for row in log_rows))

print("\nFirst 10 entries:")
for row in log_rows[:10]:
    print(row)

Cleaning log created: /content/nimbuspay_assignment/candidate_pack/cleaning_log.csv
Total log entries: 155
Unique records documented: 155

Reason counts:
Counter({'Renamed': 70, 'Removed': 40, 'Reconstructed': 25, 'Normalized': 20})

First 10 entries:
['tr-00020', 'fix', 'Renamed legacy prio field to priority and normalized its value.']
['tr-00049', 'fix', 'Renamed legacy prio field to priority and normalized its value.']
['tr-00058', 'fix', 'Renamed legacy prio field to priority and normalized its value.']
['tr-00060', 'fix', 'Reconstructed malformed or truncated assistant JSON label using the assignment schema.']
['tr-00071', 'fix', 'Normalized transaction ID to uppercase NP followed by 10 digits.']
['tr-00072', 'fix', 'Renamed legacy prio field to priority and normalized its value.']
['tr-00077', 'fix', 'Renamed legacy prio field to priority and normalized its value.']
['tr-00082', 'fix', "Removed 22 quoted support-history lines while preserving the customer's message."]
['tr-00095'

In [ ]:
#Verify quoted-history logging
import json

raw_records = read_jsonl(RAW_PATH)

quoted_record_ids = set()

for record in raw_records:
    user_text = get_message(record, "user")
    if any(line.lstrip().startswith(">") for line in user_text.splitlines()):
        quoted_record_ids.add(record["id"])

logged_quote_ids = {
    row[0] for row in log_rows
    if row[2].startswith("Removed")
}

print("Original records containing quoted lines:", len(quoted_record_ids))
print("Records documented for quote removal:", len(logged_quote_ids))
print("Missing records:", len(quoted_record_ids - logged_quote_ids))
print("Extra records:", len(logged_quote_ids - quoted_record_ids))

print("Missing IDs:", sorted(quoted_record_ids - logged_quote_ids)[:20])

Original records containing quoted lines: 40
Records documented for quote removal: 40
Missing records: 0
Extra records: 0
Missing IDs: []


In [ ]:
#Data preparation
import os

print("Assignment folder contents:\n")

for item in sorted(os.listdir(BASE)):
    path = os.path.join(BASE, item)
    print(item, " [folder]" if os.path.isdir(path) else " [file]")

print("\nScoring files:")
for root, dirs, files in os.walk(BASE):
    dirs[:] = [d for d in dirs if d != ".git"]
    for file in files:
        if file.lower() in ["score.py", "schema.md", "assignment.md"]:
            print(os.path.join(root, file))

Assignment folder contents:

ASSIGNMENT.md  [file]
SCHEMA.md  [file]
cleaning_log.csv  [file]
data  [folder]
score.py  [file]
train_cleaned.jsonl  [file]
train_reconstructed.jsonl  [file]

Scoring files:
/content/nimbuspay_assignment/candidate_pack/ASSIGNMENT.md
/content/nimbuspay_assignment/candidate_pack/score.py
/content/nimbuspay_assignment/candidate_pack/SCHEMA.md


In [ ]:
#Read score.py
SCORE_PATH = os.path.join(BASE, "score.py")

with open(SCORE_PATH, "r", encoding="utf-8") as f:
    score_code = f.read()

print(score_code)

"""Score triage predictions against labelled data.

    python score.py --gold data/dev.jsonl --pred my_dev_predictions.jsonl

Prediction file: one JSON object per line
    {"id": "dv-00001", "output": "<the raw text your model generated>"}

Scoring is strict:
  * "output" must be ONLY a JSON object (leading/trailing whitespace is ignored).
  * Every field must match exactly, including type (true is not 1, 129900 is not "129900").
  * A missing id or invalid JSON counts as wrong on every field.
"""
import argparse, json
from collections import defaultdict

KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]


def load(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def same(a, b):
    return type(a) is type(b) and a == b


def parse(text):
    try:
        obj = json.loads(text.strip())
    except (json.JSONDecodeError, AttributeError):
        return None
    re

In [ ]:
#Read the schem
SCHEMA_PATH = os.path.join(BASE, "SCHEMA.md")

with open(SCHEMA_PATH, "r", encoding="utf-8") as f:
    schema_text = f.read()

print(schema_text)

# NimbusPay Triage Labelling Spec (v2, current)

NimbusPay is a fictional payments app. Every incoming support ticket is turned into one JSON object, the *triage record*. This document is the single source of truth for what a correct record looks like. **All dev and test labels follow this spec exactly.**

## Input

```
Received: 2026-08-14 (Fri)
Source: email

<the customer's message>
```

`Received` is the date the ticket arrived. `Source` is one of `app-chat`, `email`, `twitter`, `ivr-transcript`.

## Output

A single JSON object with **exactly these 8 keys**. Use `null` when a value is not present in the ticket.

```json
{"category": "refund", "priority": "P2", "amount_paise": 2936300, "txn_date": "2026-09-13", "txn_id": "NP5972758064", "channel": "netbanking", "language": "en", "needs_human": false}
```

| Key | Type | Allowed values |
|---|---|---|
| `category` | string | `payment_failed`, `refund`, `fraud`, `account_access`, `kyc`, `offers`, `other` |
| `priority` | string | `P1

In [ ]:
#Check the development dataset format
import json
import os

DEV_PATH = os.path.join(DATA, "dev.jsonl")

with open(DEV_PATH, "r", encoding="utf-8") as f:
    dev_records = [json.loads(line) for line in f if line.strip()]

print("Development records:", len(dev_records))
print("\nFirst development record:")
print(json.dumps(dev_records[0], indent=2, ensure_ascii=False))

print("\nMessage roles:")
for message in dev_records[0]["messages"]:
    print(message["role"], ":", message["content"][:300])

print("\nAvailable slices:", dev_records[0].get("slice", "No slice field"))

Development records: 200

First development record:
{
  "id": "dv-00001",
  "messages": [
    {
      "role": "system",
      "content": "You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON."
    },
    {
      "role": "user",
      "content": "Received: 2026-08-28 (Fri)\nSource: app-chat\n\nGood morning, What needs to be done to activate and use a card? My earlier ticket number is #54047. Please look into it."
    },
    {
      "role": "assistant",
      "content": "{\"category\": \"other\", \"priority\": \"P3\", \"amount_paise\": null, \"txn_date\": null, \"txn_id\": null, \"channel\": null, \"language\": \"en\", \"needs_human\": false}"
    }
  ]
}

Message roles:
system : You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.
user : Received: 2026-08-28 (Fri)
Source: app-chat

Good morning, What needs to be done to activate and use a card? My earlier ticket number is #54047. Please look into it.
assistant : {"category":

In [ ]:
#Inspect one development record
print(json.dumps(dev_records[0], indent=2, ensure_ascii=False))

{
  "id": "dv-00001",
  "messages": [
    {
      "role": "system",
      "content": "You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON."
    },
    {
      "role": "user",
      "content": "Received: 2026-08-28 (Fri)\nSource: app-chat\n\nGood morning, What needs to be done to activate and use a card? My earlier ticket number is #54047. Please look into it."
    },
    {
      "role": "assistant",
      "content": "{\"category\": \"other\", \"priority\": \"P3\", \"amount_paise\": null, \"txn_date\": null, \"txn_id\": null, \"channel\": null, \"language\": \"en\", \"needs_human\": false}"
    }
  ]
}


### Preserved results from your completed preprocessing work

The uploaded notebook recorded:
- 1,445 training records, 200 development records, and 400 test inputs.
- 25 malformed/truncated assistant labels reconstructed.
- 70 legacy `prio` keys renamed to `priority`.
- 20 transaction IDs normalized to `NP` plus 10 digits.
- 1,024 quoted-history lines removed from 63 reconstructed records, with all 1,445 records retained.
- Final cleaned token lengths: median 167, p95 218, p99 235, maximum 273; zero records above 512 tokens.
- Assistant labels unchanged during quote removal; verification passed.
- `cleaning_log.csv`: 155 entries across 155 unique records (70 renamed, 40 quoted-history removals, 25 reconstructed, 20 normalized).

The initial quality check before the 20 transaction-ID normalizations reported 20 invalid IDs; the later normalization cell fixes those. The final audit below rechecks the completed file.

## Part A — Final schema and cleaning-log verification

In [ ]:
import json, re, csv, os
from datetime import datetime
from collections import Counter

EXPECTED_KEYS = [
    "category", "priority", "amount_paise", "txn_date",
    "txn_id", "channel", "language", "needs_human"
]
VALID_CATEGORIES = {"payment_failed","refund","fraud","account_access","kyc","offers","other"}
VALID_CHANNELS = {"upi","card","netbanking","wallet",None}
VALID_PRIORITIES = {"P1","P2","P3"}

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_records = read_jsonl(CLEAN_PATH)
dev_records = read_jsonl(os.path.join(DATA, "dev.jsonl"))
test_records = read_jsonl(os.path.join(DATA, "test_inputs.jsonl"))

audit_errors = []
for rec in train_records:
    assistant = next(m["content"] for m in rec["messages"] if m["role"]=="assistant")
    try:
        y = json.loads(assistant)
    except Exception as exc:
        audit_errors.append((rec["id"], "invalid JSON", str(exc)))
        continue
    if set(y) != set(EXPECTED_KEYS):
        audit_errors.append((rec["id"], "key mismatch", list(y)))
        continue
    if y["category"] not in VALID_CATEGORIES: audit_errors.append((rec["id"], "category"))
    if y["priority"] not in VALID_PRIORITIES: audit_errors.append((rec["id"], "priority"))
    if y["channel"] not in VALID_CHANNELS: audit_errors.append((rec["id"], "channel"))
    if not isinstance(y["needs_human"], bool): audit_errors.append((rec["id"], "needs_human"))
    if y["amount_paise"] is not None and (type(y["amount_paise"]) is not int or y["amount_paise"] < 0):
        audit_errors.append((rec["id"], "amount_paise"))
    if y["txn_date"] is not None:
        try: datetime.strptime(y["txn_date"], "%Y-%m-%d")
        except ValueError: audit_errors.append((rec["id"], "txn_date"))
    if y["txn_id"] is not None and not re.fullmatch(r"NP\d{10}", y["txn_id"]):
        audit_errors.append((rec["id"], "txn_id"))
    if y["category"] in {"account_access","kyc","other"}:
        if any(y[k] is not None for k in ["amount_paise","txn_date","txn_id","channel"]):
            audit_errors.append((rec["id"], "transaction fields must be null"))

print("Train / dev / test:", len(train_records), len(dev_records), len(test_records))
print("Final training schema errors:", len(audit_errors))
print("First errors:", audit_errors[:10])
assert len(train_records)==1445 and len(dev_records)==200 and len(test_records)==400
assert not audit_errors, "Resolve final schema errors before training."

LOG_PATH = os.path.join(BASE, "cleaning_log.csv")
with open(LOG_PATH, newline="", encoding="utf-8") as f:
    log_rows = list(csv.DictReader(f))
print("Cleaning log entries:", len(log_rows))
print("Cleaning log columns:", list(log_rows[0].keys()) if log_rows else [])
assert set(["id","action","reason"]).issubset(log_rows[0].keys())

## Part B — Baselines

Both baselines use the base model with greedy decoding and `max_new_tokens=200`. The first uses the original system/user messages. The second adds the assignment schema guidance to the system prompt. **The baseline bug in the uploaded notebook is fixed:** `apply_chat_template(..., return_tensors="pt")` returns a `BatchEncoding`; this notebook correctly passes its `input_ids` tensor to `generate()`.

### Historical baseline outputs from the earlier Colab run

The following values are the actual baseline scores previously shared in this conversation from an earlier Colab run. They are included as **historical results**, not as outputs from executing this copy of the notebook. The baseline cells below regenerate the files and scores in the current runtime.

| Metric | Baseline 1: original prompt | Baseline 2: schema guidance |
|---|---:|---:|
| Development records | 200 | 200 |
| Valid JSON | 0.0% | 95.5% |
| Exact match | 0.0% | 0.5% |
| Mean field accuracy | 0.0% | 57.2% |

**Baseline 2 per-field accuracy:**

| Field | Accuracy |
|---|---:|
| category | 62.5% |
| priority | 35.5% |
| amount_paise | 77.0% |
| txn_date | 66.5% |
| txn_id | 60.0% |
| channel | 62.0% |
| language | 63.5% |
| needs_human | 30.5% |

**Interpretation:** adding explicit schema guidance substantially improved JSON validity and field accuracy, but exact-match performance remained very low. `needs_human` and `priority` were especially weak and motivate the fine-tuning and targeted error analysis.

**Integrity note:** the historical baseline prediction files in the assignment ZIP were empty. These numbers are recorded here from the earlier run described in the conversation; they are not recreated prediction files. Run the baseline cells to produce fresh `baseline1_dev_predictions.jsonl`, `baseline2_dev_predictions.jsonl`, and `baseline_metrics.json`.


In [ ]:
# Load the base model and tokenizer for both baselines
import torch, gc, json, os, time
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
base_model.eval()
print("Loaded:", MODEL_ID)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Official scoring function from the supplied score.py
import importlib.util
spec = importlib.util.spec_from_file_location("official_score", SCORE_PATH)
official_score = importlib.util.module_from_spec(spec)
spec.loader.exec_module(official_score)

def score_predictions(gold_rows, pred_rows):
    return official_score.score(gold_rows, pred_rows)

def make_inference_messages(record, add_schema=False):
    messages = [
        {"role": m["role"], "content": m["content"]}
        for m in record["messages"]
        if m["role"] in ("system", "user")
    ]
    if add_schema:
        for msg in messages:
            if msg["role"] == "system":
                msg["content"] += (
                    "\n\nFollow this NimbusPay schema exactly:\n"
                    + schema_text
                    + "\nReturn only one JSON object with exactly the required 8 keys. "
                      "Do not use Markdown or additional text."
                )
                break
    return messages

def generate_one(model, record, add_schema=False):
    msgs = make_inference_messages(record, add_schema=add_schema)
    encoded = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    )
    # return_dict=True guarantees a BatchEncoding with an input_ids field.
    input_device = model.get_input_embeddings().weight.device
    input_ids = encoded["input_ids"].to(input_device)
    with torch.inference_mode():
        out = model.generate(
            input_ids,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = out[0, input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def save_predictions(path, predictions):
    with open(path, "w", encoding="utf-8") as f:
        for row in predictions:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def run_baseline(name, add_schema=False):
    path = os.path.join(BASE, f"{name}_dev_predictions.jsonl")
    preds = []
    for rec in tqdm(dev_records, desc=name):
        preds.append({"id": rec["id"], "output": generate_one(base_model, rec, add_schema)})
    save_predictions(path, preds)
    metrics = score_predictions(dev_records, preds)
    print(name, "saved:", path)
    print(json.dumps(metrics["all"], indent=2))
    return preds, metrics

baseline1_predictions, baseline1_metrics = run_baseline("baseline1", add_schema=False)
baseline2_predictions, baseline2_metrics = run_baseline("baseline2", add_schema=True)

In [ ]:
# Save baseline metrics and release the base model before fine-tuning
baseline_summary = {
    "baseline1_original_prompt": baseline1_metrics["all"],
    "baseline2_schema_prompt": baseline2_metrics["all"]
}
with open(os.path.join(BASE, "baseline_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(baseline_summary, f, indent=2)
print("Baseline metrics saved.")
del base_model
gc.collect()
torch.cuda.empty_cache()

## Part C — Fine-tuning method

- Base model: Qwen2.5-1.5B-Instruct (under the 4B limit).
- FP16 LoRA reference: rank 16, alpha 32, dropout 0.05; target attention and MLP projection layers.
- Assistant-only loss: system and user tokens are masked with `-100`; only assistant completion tokens contribute to loss.
- Maximum sequence length: 512. The cleaned training data has p95=218 and max=273 tokens, so 512 retains the observed cleaned examples with margin.
- Effective batch size: micro-batch 2 × gradient accumulation 8 = 16.
- Optimizer: AdamW, learning rate 2e-4, weight decay 0.01, 2 epochs, gradient clipping 1.0.
- LoRA trainable parameters are counted and asserted below 50M.

### Historical FP16 training result preserved from your uploaded notebook

Your uploaded notebook contained a successful earlier FP16 LoRA training run:

| Metric | Recorded output |
|---|---:|
| Epoch 1 average loss | 0.1078 |
| Epoch 2 average loss | 0.0152 |
| Training time | 9.80 minutes |
| Peak GPU memory | 4.93 GB |
| Trainable parameters | 18,464,768 (1.182%) |

This is retained as historical output, not treated as a current adapter file. The next cell reruns training so the adapter is actually present in the current Colab session and submission folder.

### Historical runtime compatibility note

The earlier FP16 run used Qwen/Qwen2.5-1.5B-Instruct with LoRA rank 16, alpha 32, dropout 0.05, and attention plus MLP projection targets. It completed two epochs in 9.80 minutes with 4.93 GB peak GPU memory and 18,464,768 trainable parameters (1.182%). These are historical measurements; rerun the reference-training cell to recreate the adapter in the current runtime.

The earlier Colab environment reported PyTorch 2.14.1+cu130 and bitsandbytes 0.46.1, but bitsandbytes did not provide a CUDA 13.0 binary (`libbitsandbytes_cuda130.so`). Therefore there is **no verified QLoRA score** from that run. The QLoRA cell in this notebook attempts the comparison and records an error if the installed stack is incompatible; do not report a QLoRA metric unless that cell succeeds.


In [ ]:
# Assistant-only dataset and reusable training / evaluation utilities
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_
from peft import LoraConfig, TaskType, get_peft_model
import time, gc, json, os, math

MAX_LENGTH = 512
BATCH_SIZE = 2
GRAD_ACCUM = 8
EPOCHS = 2
LEARNING_RATE = 2e-4
LORA_R = 16
LORA_ALPHA = 32
ALL_TARGET_MODULES = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
ATTN_TARGET_MODULES = ["q_proj","k_proj","v_proj","o_proj"]

class NimbusDataset(Dataset):
    def __init__(self, records, tokenizer, max_length=MAX_LENGTH):
        self.examples = []
        zero_supervision = []
        for rec in records:
            msgs = rec["messages"]
            full_ids = tokenizer.apply_chat_template(
                msgs, tokenize=True, add_generation_prompt=False
            )
            prefix_ids = tokenizer.apply_chat_template(
                msgs[:-1], tokenize=True, add_generation_prompt=True
            )
            if full_ids[:len(prefix_ids)] != prefix_ids:
                raise ValueError(f"Chat-template prefix mismatch: {rec['id']}")
            input_ids = full_ids[:max_length]
            labels = input_ids.copy()
            mask_len = min(len(prefix_ids), len(labels))
            labels[:mask_len] = [-100] * mask_len
            supervised = sum(x != -100 for x in labels)
            if supervised == 0:
                zero_supervision.append(rec["id"])
            self.examples.append({"input_ids": input_ids, "labels": labels})
        if zero_supervision:
            raise ValueError(f"{len(zero_supervision)} examples have no assistant tokens after truncation; first IDs: {zero_supervision[:10]}")

    def __len__(self): return len(self.examples)
    def __getitem__(self, idx): return self.examples[idx]

def collate_nimbus(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    ids, labels, masks = [], [], []
    for item in batch:
        pad = max_len - len(item["input_ids"])
        ids.append(item["input_ids"] + [tokenizer.pad_token_id] * pad)
        labels.append(item["labels"] + [-100] * pad)
        masks.append([1] * len(item["input_ids"]) + [0] * pad)
    return {
        "input_ids": torch.tensor(ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "attention_mask": torch.tensor(masks, dtype=torch.long)
    }

print("Training utility functions ready.")
print("Max sequence length:", MAX_LENGTH)
print("Effective batch size:", BATCH_SIZE * GRAD_ACCUM)

In [ ]:
# Experiment runner: FP16 LoRA or 4-bit QLoRA, with identical training/evaluation logic.
from peft import prepare_model_for_kbit_training

experiment_results = []
experiment_predictions = {}
experiment_adapters = {}

def load_lora_model(quantized=False, rank=LORA_R, alpha=LORA_ALPHA, target_modules=None):
    target_modules = target_modules or ALL_TARGET_MODULES
    if quantized:
        from transformers import BitsAndBytesConfig
        qconfig = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )
        base = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=qconfig,
            device_map="auto",
            trust_remote_code=True
        )
        base = prepare_model_for_kbit_training(
            base, use_gradient_checkpointing=True
        )
    else:
        base = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True
        )
        base.config.use_cache = False
        base.gradient_checkpointing_enable()
        base.enable_input_require_grads()

    config = LoraConfig(
        r=rank,
        lora_alpha=alpha,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=target_modules
    )
    wrapped = get_peft_model(base, config)
    wrapped.config.use_cache = False
    return wrapped

def run_experiment(name, records, rank=LORA_R, alpha=LORA_ALPHA,
                   lr=LEARNING_RATE, epochs=EPOCHS,
                   target_modules=None, quantized=False):
    print("\n" + "="*72)
    print("EXPERIMENT:", name)
    print("Data records:", len(records), "| rank:", rank, "| lr:", lr,
          "| epochs:", epochs, "| quantized:", quantized)
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model = load_lora_model(
        quantized=quantized, rank=rank, alpha=alpha,
        target_modules=target_modules or ALL_TARGET_MODULES
    )
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.3f}%)")
    assert trainable <= 50_000_000, "Trainable parameter budget exceeded."

    dataset = NimbusDataset(records, tokenizer, MAX_LENGTH)
    loader = DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=collate_nimbus
    )
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr, weight_decay=0.01
    )
    scaler = torch.amp.GradScaler("cuda") if not quantized else torch.amp.GradScaler("cuda")
    model.train()
    optimizer.zero_grad(set_to_none=True)
    losses = []
    started = time.time()

    for epoch in range(epochs):
        epoch_loss = 0.0
        for step, batch in enumerate(tqdm(loader, desc=f"{name} epoch {epoch+1}/{epochs}")):
            group_start = (step // GRAD_ACCUM) * GRAD_ACCUM
            group_size = min(GRAD_ACCUM, len(loader) - group_start)
            batch = {k: v.to(model.device) for k,v in batch.items()}
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(**batch).loss
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss in {name}, epoch {epoch+1}, step {step+1}")
            scaler.scale(loss / group_size).backward()
            epoch_loss += float(loss.detach().cpu())
            if (step+1) % GRAD_ACCUM == 0 or (step+1) == len(loader):
                scaler.unscale_(optimizer)
                clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0
                )
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
        mean_loss = epoch_loss / len(loader)
        losses.append(mean_loss)
        print(f"{name} epoch {epoch+1} mean loss: {mean_loss:.6f}")

    train_minutes = (time.time() - started) / 60
    peak_gb = torch.cuda.max_memory_allocated() / 1024**3
    adapter_dir = os.path.join(BASE, f"adapter_{name}")
    os.makedirs(adapter_dir, exist_ok=True)
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)

    model.eval()
    preds = []
    pred_path = os.path.join(BASE, f"{name}_dev_predictions.jsonl")
    with open(pred_path, "w", encoding="utf-8") as f:
        for rec in tqdm(dev_records, desc=f"{name} dev evaluation"):
            output = generate_one(model, rec, add_schema=False)
            row = {"id": rec["id"], "output": output}
            preds.append(row)
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    metrics = score_predictions(dev_records, preds)
    all_metrics = metrics["all"]
    result = {
        "name": name,
        "quantized_4bit": bool(quantized),
        "rank": rank,
        "alpha": alpha,
        "learning_rate": lr,
        "epochs": epochs,
        "target_modules": target_modules or ALL_TARGET_MODULES,
        "trainable_parameters": trainable,
        "loss_by_epoch": losses,
        "training_minutes": round(train_minutes, 2),
        "peak_gpu_memory_gb": round(peak_gb, 2),
        "json_valid": all_metrics["json_valid"],
        "exact_match": all_metrics["exact_match"],
        "mean_field_acc": all_metrics["mean_field_acc"],
        "field_accuracy": all_metrics["fields"],
        "adapter_dir": adapter_dir,
        "dev_predictions": pred_path,
        "status": "success"
    }
    experiment_results.append(result)
    experiment_predictions[name] = preds
    experiment_adapters[name] = {"path": adapter_dir, "quantized": bool(quantized)}
    print(json.dumps(result, indent=2))

    del model, optimizer, loader, dataset
    gc.collect()
    torch.cuda.empty_cache()
    return result

### Reference model — cleaned data, FP16 LoRA

In [ ]:
reference_result = run_experiment(
    name="fp16_reference_cleaned",
    records=train_records,
    rank=16, alpha=32, lr=2e-4, epochs=2,
    target_modules=ALL_TARGET_MODULES,
    quantized=False
)

## Part D — Controlled experiments

Each comparison changes one main factor while keeping the other reference settings fixed:

1. **Raw/reconstructed vs cleaned data:** same FP16 LoRA settings, different training-data version.
2. **LoRA rank 8 vs rank 16:** same cleaned data, FP16, target modules, learning rate and epochs.
3. **Attention-only vs attention+MLP target modules:** same cleaned data, rank, learning rate and epochs.
4. **FP16 LoRA vs 4-bit QLoRA:** same cleaned data, rank, target modules, learning rate and epochs; only base-model loading/compute quantization changes.

The QLoRA cell attempts to install and verify bitsandbytes. If the current Colab PyTorch/CUDA build is unsupported, the experiment is marked **failed**, not given fabricated metrics. In that case, the FP16 experiments still run, but the assignment's QLoRA comparison remains to be completed on a compatible Colab runtime.

In [ ]:
# Experiment 1 — reconstructed/raw messages vs cleaned messages (only data version changes)
reconstructed_records = read_jsonl(os.path.join(BASE, "train_reconstructed.jsonl"))
raw_data_result = run_experiment(
    name="fp16_reconstructed_data",
    records=reconstructed_records,
    rank=16, alpha=32, lr=2e-4, epochs=2,
    target_modules=ALL_TARGET_MODULES,
    quantized=False
)

In [ ]:
# Experiment 2 — LoRA rank 8 vs reference rank 16 (rank is the changed factor)
rank8_result = run_experiment(
    name="fp16_rank8",
    records=train_records,
    rank=8, alpha=32, lr=2e-4, epochs=2,
    target_modules=ALL_TARGET_MODULES,
    quantized=False
)

In [ ]:
# Experiment 3 — attention-only targets vs reference attention+MLP targets
attention_only_result = run_experiment(
    name="fp16_attention_only",
    records=train_records,
    rank=16, alpha=32, lr=2e-4, epochs=2,
    target_modules=ATTN_TARGET_MODULES,
    quantized=False
)

In [ ]:
# Experiment 4 — 4-bit QLoRA vs FP16 LoRA (same data and LoRA hyperparameters)
qlora_result = None
qlora_status = {"status": "not_run", "reason": None}
try:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'bitsandbytes'], check=True)
    import bitsandbytes as bnb
    print("bitsandbytes:", bnb.__version__)
    qlora_result = run_experiment(
        name="qlora4bit_cleaned",
        records=train_records,
        rank=16, alpha=32, lr=2e-4, epochs=2,
        target_modules=ALL_TARGET_MODULES,
        quantized=True
    )
    qlora_status = {"status": "success", "reason": None}
except Exception as exc:
    qlora_status = {"status": "failed", "reason": repr(exc)}
    print("QLoRA experiment failed on this runtime:", repr(exc))
    print("Do not report QLoRA metrics unless this experiment succeeds.")
finally:
    with open(os.path.join(BASE, "qlora_status.json"), "w", encoding="utf-8") as f:
        json.dump(qlora_status, f, indent=2)

In [ ]:
# Consolidate actual experiment metrics (no placeholder values)
import pandas as pd
results_df = pd.DataFrame([
    {
        "experiment": r["name"],
        "status": r["status"],
        "quantized_4bit": r["quantized_4bit"],
        "rank": r["rank"],
        "learning_rate": r["learning_rate"],
        "epochs": r["epochs"],
        "trainable_parameters": r["trainable_parameters"],
        "dev_exact_match": r["exact_match"],
        "dev_mean_field_accuracy": r["mean_field_acc"],
        "peak_gpu_memory_gb": r["peak_gpu_memory_gb"],
        "training_minutes": r["training_minutes"]
    }
    for r in experiment_results
])
display(results_df)
results_df.to_csv(os.path.join(BASE, "experiment_results.csv"), index=False)
with open(os.path.join(BASE, "experiment_results.json"), "w", encoding="utf-8") as f:
    json.dump(experiment_results, f, indent=2)

## Part E — Best model selection and detailed development evaluation

In [ ]:
# Select the best successful model by dev exact match, then mean field accuracy.
if not experiment_results:
    raise RuntimeError("No fine-tuning experiment completed successfully.")

best_result = sorted(
    experiment_results,
    key=lambda r: (r["exact_match"], r["mean_field_acc"]),
    reverse=True
)[0]
BEST_NAME = best_result["name"]
BEST_ADAPTER_DIR = best_result["adapter_dir"]
BEST_IS_4BIT = best_result["quantized_4bit"]

print("Best model:", BEST_NAME)
print("Dev exact match:", f"{best_result['exact_match']:.2%}")
print("Dev mean field accuracy:", f"{best_result['mean_field_acc']:.2%}")
print("4-bit QLoRA:", BEST_IS_4BIT)
print("Per-field accuracy:")
for field, value in best_result["field_accuracy"].items():
    print(f"  {field:16s} {value:.2%}")

with open(os.path.join(BASE, "best_model.json"), "w", encoding="utf-8") as f:
    json.dump(best_result, f, indent=2)

In [ ]:
# Load the selected adapter for error analysis and test inference
from peft import PeftModel
from transformers import BitsAndBytesConfig

gc.collect()
torch.cuda.empty_cache()

if BEST_IS_4BIT:
    best_qconfig = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    best_base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=best_qconfig,
        device_map="auto", trust_remote_code=True
    )
else:
    best_base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16,
        device_map="auto", trust_remote_code=True
    )

best_model = PeftModel.from_pretrained(best_base, BEST_ADAPTER_DIR)
best_model.eval()
print("Best adapter loaded:", BEST_ADAPTER_DIR)

### Error analysis — weakest fields, categories, and at least 10 concrete failures

In [ ]:
# Generate the best model's dev predictions again from the selected adapter.
best_dev_predictions = []
best_dev_path = os.path.join(BASE, "best_model_dev_predictions.jsonl")
with open(best_dev_path, "w", encoding="utf-8") as f:
    for rec in tqdm(dev_records, desc="Best model dev predictions"):
        row = {"id": rec["id"], "output": generate_one(best_model, rec)}
        best_dev_predictions.append(row)
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

best_dev_metrics = score_predictions(dev_records, best_dev_predictions)
print("Best model overall metrics:")
print(json.dumps(best_dev_metrics["all"], indent=2))

# Per-category exact match and per-field accuracy.
category_summary = {
    key: value for key, value in best_dev_metrics.items()
    if key.startswith("category=")
}
print("\nCategory exact-match summary:")
for key, value in sorted(category_summary.items()):
    print(key, "n=", value["n"], "exact=", f"{value['exact_match']:.2%}",
          "field_acc=", f"{value['mean_field_acc']:.2%}")

field_acc = best_dev_metrics["all"]["fields"]
print("\nWeakest fields:")
for field, value in sorted(field_acc.items(), key=lambda kv: kv[1]):
    print(f"{field:16s} {value:.2%}")

In [ ]:
# Build a reviewable error-analysis CSV with 10+ specific failures and evidence-based hypotheses.
gold_by_id = {}
for rec in dev_records:
    gold_by_id[rec["id"]] = json.loads(
        next(m["content"] for m in rec["messages"] if m["role"]=="assistant")
    )
pred_by_id = {p["id"]: p["output"] for p in best_dev_predictions}

def hypothesis_for(field):
    return {
        "category": "Review the ticket's primary intent and the schema's category boundaries; fraud overrides other categories when the transaction is unrecognized.",
        "priority": "Check the P1 fraud / disputed-amount / legal-threat rules first, then the P2 amount and account-access rules.",
        "amount_paise": "Check whether the model selected the amount the ticket is about, ignored balances/fees, and converted rupees to paise correctly.",
        "txn_date": "Check Received-date anchoring for relative dates and ensure quoted earlier-email dates are ignored.",
        "txn_id": "Check whether the value is an NP plus 10-digit NimbusPay ID rather than a bank, order, or ticket reference.",
        "channel": "Check that the payment mode is explicitly stated and mapped to upi/card/netbanking/wallet.",
        "language": "Check whether the customer's message is Hinglish written in Latin script.",
        "needs_human": "Check whether P1 or prior/repeated support contact requires needs_human=true."
    }.get(field, "Review the ticket and the corresponding schema rule.")

error_rows = []
for rec in dev_records:
    rid = rec["id"]
    gold = gold_by_id[rid]
    raw = pred_by_id.get(rid, "")
    try:
        pred = json.loads(raw)
        valid_json = isinstance(pred, dict)
    except Exception:
        pred, valid_json = {}, False

    user_text = next(m["content"] for m in rec["messages"] if m["role"]=="user")
    if not valid_json:
        error_rows.append({
            "id": rid, "field": "JSON",
            "gold_value": json.dumps(gold, ensure_ascii=False),
            "predicted_value": raw,
            "ticket_excerpt": user_text[:500],
            "observed_difference": "Output is not a valid JSON object.",
            "diagnostic_hypothesis": "Review instruction following, exact JSON formatting, and whether the model emitted any non-JSON text.",
            "human_review": ""
        })
        continue

    for field in EXPECTED_KEYS:
        if type(pred.get(field)) is not type(gold[field]) or pred.get(field) != gold[field]:
            error_rows.append({
                "id": rid, "field": field,
                "gold_value": json.dumps(gold[field], ensure_ascii=False),
                "predicted_value": json.dumps(pred.get(field), ensure_ascii=False),
                "ticket_excerpt": user_text[:500],
                "observed_difference": f"Gold={gold[field]!r}; prediction={pred.get(field)!r}",
                "diagnostic_hypothesis": hypothesis_for(field),
                "human_review": ""
            })

error_rows = error_rows[:max(10, min(len(error_rows), 50))]
error_path = os.path.join(BASE, "error_analysis_review.csv")
pd.DataFrame(error_rows).to_csv(error_path, index=False)
print("Specific mismatch rows exported:", len(error_rows))
print("Review file:", error_path)
display(pd.DataFrame(error_rows).head(15))
if len(error_rows) < 10:
    print("Fewer than 10 mismatches were found; inspect all available failures and add near-miss qualitative examples if the brief requires 10.")

### What to do next week

Use the observed weakest fields and category slices to choose targeted data improvements: add balanced examples for weak categories, add hard negatives for similar intents, expand relative-date and quoted-history examples, strengthen amount/paise and reference-ID examples, and review escalation cues for priority and `needs_human`. Keep a held-out evaluation set unchanged when iterating.

## Hidden test — generate 400 raw model outputs

In [ ]:
# Final inference uses only the original system/user messages, greedy decoding,
# max_new_tokens=200, and whitespace stripping. No post-hoc field correction.
test_predictions = []
TEST_PRED_PATH = os.path.join(BASE, "predictions_test.jsonl")

with open(TEST_PRED_PATH, "w", encoding="utf-8") as f:
    for rec in tqdm(test_records, desc="Hidden test predictions"):
        output = generate_one(best_model, rec, add_schema=False)
        row = {"id": rec["id"], "output": output}
        test_predictions.append(row)
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Test rows:", len(test_predictions))
print("Unique IDs:", len({x["id"] for x in test_predictions}))
print("Saved:", TEST_PRED_PATH)
print("First prediction:")
print(json.dumps(test_predictions[0], indent=2, ensure_ascii=False))
assert len(test_predictions) == 400
assert len({x["id"] for x in test_predictions}) == 400

## Reproducibility, adapter validation, README, report, and submission bundle

In [ ]:
# Save environment and verify the chosen PEFT adapter files.
import platform, subprocess, shutil, glob, json, os, time
import torch, transformers, peft

adapter_files = sorted(os.listdir(BEST_ADAPTER_DIR))
print("Adapter directory:", BEST_ADAPTER_DIR)
print("Adapter files:", adapter_files)
assert "adapter_config.json" in adapter_files
assert any(name.endswith(".safetensors") or name.endswith(".bin") for name in adapter_files)

environment = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "model_id": MODEL_ID
}
with open(os.path.join(BASE, "environment.json"), "w", encoding="utf-8") as f:
    json.dump(environment, f, indent=2)
print(json.dumps(environment, indent=2))

In [ ]:
# Create README.md using measured results only.
qlora_ok = any(r["quantized_4bit"] for r in experiment_results)
readme = f'''# NimbusPay Support Ticket Triage — LLM Fine-Tuning

## Model
- Base model: `{MODEL_ID}`
- Best adapter: `{BEST_NAME}`
- Best model uses 4-bit QLoRA: `{"Yes" if BEST_IS_4BIT else "No"}`
- Fine-tuning method: PEFT LoRA / QLoRA; assistant-only loss
- Best dev exact match: {best_result["exact_match"]:.2%}
- Best dev mean field accuracy: {best_result["mean_field_acc"]:.2%}

## One-line result
Fine-tuned Qwen2.5-1.5B-Instruct with { "4-bit QLoRA" if BEST_IS_4BIT else "FP16 LoRA" } on 1,445 cleaned NimbusPay tickets; dev exact match {best_result["exact_match"]:.2%} and mean field accuracy {best_result["mean_field_acc"]:.2%}.

## Files
- `adapter/` — PEFT adapter files only (base model is not included)
- `NimbusPay_LLM_Fine_Tuning_Final.ipynb` — upload the executed notebook separately to the GitHub repository
- `report.md` — concise experiment and error-analysis report
- `cleaning_log.csv` — row-level cleaning actions
- `predictions_test.jsonl` — one raw model output per hidden-test input
- `experiment_results.csv` — measured experiment metrics

## Reproduction
Run the notebook in Google Colab with a Tesla T4 GPU. Upload the supplied candidate pack ZIP when prompted. The notebook downloads the base model from Hugging Face and saves the adapter and submission artifacts under the assignment folder.

## QLoRA status
{json.dumps(qlora_status, ensure_ascii=False)}
'''
with open(os.path.join(BASE, "README.md"), "w", encoding="utf-8") as f:
    f.write(readme)
print(readme)

In [ ]:
# Create a concise report (designed to fit within the 3-page limit when exported).
rows_md = []
for r in experiment_results:
    rows_md.append(
        f'| {r["name"]} | {r["status"]} | {r["exact_match"]:.2%} | '
        f'{r["mean_field_acc"]:.2%} | {r["peak_gpu_memory_gb"]:.2f} | '
        f'{r["training_minutes"]:.2f} |'
    )
experiment_table = "\n".join(rows_md)

weak_fields = sorted(best_result["field_accuracy"].items(), key=lambda kv: kv[1])
category_lines = []
for key, val in sorted(category_summary.items()):
    category_lines.append(f'- {key}: n={val["n"]}, exact match={val["exact_match"]:.2%}, mean field accuracy={val["mean_field_acc"]:.2%}')

report = f'''# NimbusPay LLM Fine-Tuning Report

## 1. Data audit and cleaning
The dataset contains 1,445 training records, 200 development records and 400 hidden-test inputs. The audit identified 25 malformed/truncated assistant labels, 70 legacy `prio` keys, 20 non-normalized NimbusPay transaction IDs and quoted support-history lines. The cleaned dataset preserves all 1,445 records. The cleaning log contains {len(log_rows)} row-level actions. The final cleaned token distribution recorded in the notebook is median 167, p95 218 and maximum 273 tokens; therefore max sequence length 512 was selected.

## 2. Model and training
Base model: `{MODEL_ID}` (1.5B parameters). Fine-tuning uses PEFT LoRA, with assistant-only loss. Reference settings: rank 16, alpha 32, dropout 0.05, learning rate 2e-4, two epochs, micro-batch 2, gradient accumulation 8, max sequence length 512. The trainable-parameter budget is checked against 50M.

## 3. Baselines
- Original prompt baseline: JSON validity {baseline1_metrics["all"]["json_valid"]:.2%}, exact match {baseline1_metrics["all"]["exact_match"]:.2%}, mean field accuracy {baseline1_metrics["all"]["mean_field_acc"]:.2%}.
- Schema-guided prompt baseline: JSON validity {baseline2_metrics["all"]["json_valid"]:.2%}, exact match {baseline2_metrics["all"]["exact_match"]:.2%}, mean field accuracy {baseline2_metrics["all"]["mean_field_acc"]:.2%}.

## 4. Experiment results
| Experiment | Status | Dev exact match | Mean field accuracy | Peak GPU GB | Training min |
|---|---|---:|---:|---:|---:|
{experiment_table}

QLoRA status: {json.dumps(qlora_status, ensure_ascii=False)}. A failed QLoRA run is not reported as a successful comparison.

## 5. Best model and error analysis
Best successful model: `{BEST_NAME}`. Dev exact match: {best_result["exact_match"]:.2%}; mean field accuracy: {best_result["mean_field_acc"]:.2%}; 4-bit: {BEST_IS_4BIT}.
Weakest fields:
{chr(10).join(f'- {k}: {v:.2%}' for k,v in weak_fields)}
Category slices:
{chr(10).join(category_lines)}
Specific failure examples and diagnostic hypotheses are saved in `error_analysis_review.csv`; review those rows before final submission.

## 6. Next steps
Improve weak fields with targeted examples and hard negatives; add examples for relative dates, quoted histories, rupees-to-paise conversion, NimbusPay transaction-ID normalization, explicit payment-channel detection, and escalation rules. Preserve the dev set as a fixed evaluation set.

## 7. Reproducibility
Environment details are in `environment.json`. The PEFT adapter is saved separately; the base model and dataset are not bundled.
'''
with open(os.path.join(BASE, "report.md"), "w", encoding="utf-8") as f:
    f.write(report)
print("Report saved:", os.path.join(BASE, "report.md"))

In [ ]:
# Final submission checks and ZIP bundle.
required_files = [
    "cleaning_log.csv",
    "predictions_test.jsonl",
    "README.md",
    "report.md",
    "experiment_results.csv",
    "environment.json",
    "best_model.json",
    "error_analysis_review.csv",
    "adapter_" + BEST_NAME
]
missing = [name for name in required_files if not os.path.exists(os.path.join(BASE, name))]
print("Missing required items:", missing)
prediction_count = sum(1 for _ in open(TEST_PRED_PATH, encoding="utf-8"))
print("Test predictions:", prediction_count)
assert prediction_count == 400, "Expected exactly 400 test predictions."
error_review_count = sum(1 for _ in open(os.path.join(BASE, "error_analysis_review.csv"), encoding="utf-8")) - 1
print("Error-analysis mismatch rows:", error_review_count)
assert error_review_count >= 10, "Review at least 10 specific dev errors before submission."
print("Adapter config exists:", os.path.exists(os.path.join(BEST_ADAPTER_DIR, "adapter_config.json")))
print("QLoRA experiment:", qlora_status["status"])

# Bundle only submission artifacts; do not include datasets or downloaded base model.
SUBMISSION_DIR = os.path.join(BASE, "submission")
os.makedirs(SUBMISSION_DIR, exist_ok=True)

def copy_if_exists(src, dst_name=None):
    src_path = os.path.join(BASE, src)
    if os.path.isdir(src_path):
        dst_path = os.path.join(SUBMISSION_DIR, dst_name or src)
        if os.path.exists(dst_path): shutil.rmtree(dst_path)
        shutil.copytree(src_path, dst_path)
    elif os.path.isfile(src_path):
        shutil.copy2(src_path, os.path.join(SUBMISSION_DIR, dst_name or src))

copy_if_exists(BEST_ADAPTER_DIR, "adapter")
for fname in [
    "cleaning_log.csv", "predictions_test.jsonl", "README.md", "report.md",
    "experiment_results.csv", "experiment_results.json", "baseline_metrics.json",
    "best_model.json", "environment.json", "error_analysis_review.csv",
    "qlora_status.json"
]:
    copy_if_exists(fname)

ZIP_OUT = "/content/NimbusPay_submission_bundle.zip"
if os.path.exists(ZIP_OUT): os.remove(ZIP_OUT)
shutil.make_archive(ZIP_OUT[:-4], "zip", SUBMISSION_DIR)
print("Submission bundle:", ZIP_OUT)
print("Submission folder files:", sorted(os.listdir(SUBMISSION_DIR)))

In [ ]:
# Download the submission bundle to your computer.
from google.colab import files
files.download("/content/NimbusPay_submission_bundle.zip")

## Final checklist before GitHub submission

- [ ] Run every cell from top to bottom in a T4 runtime.
- [ ] Confirm final training schema audit has zero errors.
- [ ] Confirm both baselines have actual measured scores.
- [ ] Confirm the FP16 reference adapter was saved.
- [ ] Complete all four experiments; if QLoRA fails, resolve the runtime compatibility issue and rerun it before claiming the comparison is complete.
- [ ] Review at least 10 rows in `error_analysis_review.csv` and fill `human_review`.
- [ ] Confirm `predictions_test.jsonl` has exactly 400 unique IDs and raw model outputs.
- [ ] Include the executed notebook with outputs in the GitHub repository.
- [ ] Do not upload the dataset or base model weights.
- [ ] Make the repository public within the deadline window and email the repository URL as instructed in the assignment.
**Do not publish as a completed assignment until every code cell has run successfully and the generated adapter, report, error-analysis CSV, and 400-row test predictions are present. Upload this notebook file separately to the repository; the ZIP contains the supporting artifacts.**
